In [ ]:
import sys
sys.path.insert(0, '../src')

import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from model import SineMLP
from inference import OptimizedInference, benchmark_inference

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Generate Test Data

In [ ]:
def generate_cloth_mesh(resolution=50, size=2.0):
    """Generate a plane mesh for testing."""
    x = np.linspace(-size/2, size/2, resolution)
    y = np.linspace(-size/2, size/2, resolution)
    xx, yy = np.meshgrid(x, y)
    zz = np.zeros_like(xx)
    
    vertices = np.stack([xx.flatten(), yy.flatten(), zz.flatten()], axis=1)
    return vertices.astype(np.float32)

# Generate test cloth
test_vertices = generate_cloth_mesh(resolution=50)
print(f"Test mesh: {test_vertices.shape[0]} vertices")

## 2. Mass-Spring Baseline

In [ ]:
class MassSpringBaseline:
    """Simple mass-spring cloth simulator for comparison."""
    
    def __init__(self, vertices, resolution=50, stiffness=100.0, damping=0.1):
        self.n_verts = len(vertices)
        self.resolution = resolution
        self.stiffness = stiffness
        self.damping = damping
        
        self.positions = vertices.copy()
        self.velocities = np.zeros_like(vertices)
        self.rest_positions = vertices.copy()
        
        # Build edges
        self.edges = []
        for i in range(resolution):
            for j in range(resolution - 1):
                idx = i * resolution + j
                self.edges.append((idx, idx + 1))
        for i in range(resolution - 1):
            for j in range(resolution):
                idx = i * resolution + j
                self.edges.append((idx, idx + resolution))
        
        # Compute rest lengths
        self.rest_lengths = []
        for i, j in self.edges:
            length = np.linalg.norm(vertices[i] - vertices[j])
            self.rest_lengths.append(length)
        
        # Pin top row
        self.pinned = set(range(resolution))
    
    def step(self, dt=0.01, gravity=np.array([0, 0, -9.81]), wind=np.array([0, 0, 0])):
        """Simulate one time step."""
        forces = np.zeros_like(self.positions)
        
        # Gravity
        forces += gravity
        
        # Wind
        forces += wind
        
        # Spring forces
        for k, (i, j) in enumerate(self.edges):
            diff = self.positions[j] - self.positions[i]
            dist = np.linalg.norm(diff)
            if dist > 1e-8:
                direction = diff / dist
                force = self.stiffness * (dist - self.rest_lengths[k]) * direction
                forces[i] += force
                forces[j] -= force
        
        # Damping
        forces -= self.damping * self.velocities
        
        # Integration (explicit Euler)
        self.velocities += forces * dt
        self.positions += self.velocities * dt
        
        # Enforce pins
        for p in self.pinned:
            self.positions[p] = self.rest_positions[p]
            self.velocities[p] = 0
        
        return self.positions.copy()

# Test baseline
baseline = MassSpringBaseline(test_vertices)
for _ in range(100):
    baseline_result = baseline.step(wind=np.array([5, 0, 0]))

print(f"Baseline simulation complete")
print(f"Position range: Z = [{baseline_result[:, 2].min():.3f}, {baseline_result[:, 2].max():.3f}]")

## 3. Neural Model Inference

In [ ]:
# Create and trace a model (in practice, load trained model)
model = SineMLP(in_dim=8, hidden_dim=256, out_dim=3, n_layers=6)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Trace for inference
model.eval()
traced = torch.jit.trace(model, torch.randn(100, 8, device=device))
traced.save('/tmp/test_model.pt')

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load optimized inference engine
engine = OptimizedInference(
    model_path='/tmp/test_model.pt',
    use_cache=False,
    use_fp16=torch.cuda.is_available()
)

# Run inference
vertices_tensor = torch.tensor(test_vertices, dtype=torch.float32)
force = torch.tensor([5.0, 0.0, 0.0])

neural_disp = engine.predict(vertices_tensor, time=0.5, force=force, material_id=0)
neural_result = test_vertices + neural_disp.cpu().numpy()

print(f"Neural inference complete")
print(f"Displacement range: [{neural_disp.min():.4f}, {neural_disp.max():.4f}]")

## 4. Compute Metrics

In [ ]:
def chamfer_distance(points1, points2):
    """Compute Chamfer distance between two point sets."""
    from scipy.spatial import cKDTree
    
    tree1 = cKDTree(points1)
    tree2 = cKDTree(points2)
    
    dists1, _ = tree1.query(points2, k=1)
    dists2, _ = tree2.query(points1, k=1)
    
    chamfer = np.mean(dists1) + np.mean(dists2)
    return chamfer

def compute_normals(vertices, resolution):
    """Compute approximate vertex normals for a grid mesh."""
    normals = np.zeros_like(vertices)
    
    for i in range(resolution):
        for j in range(resolution):
            idx = i * resolution + j
            
            # Get neighbors
            neighbors = []
            if i > 0:
                neighbors.append(vertices[(i-1) * resolution + j])
            if i < resolution - 1:
                neighbors.append(vertices[(i+1) * resolution + j])
            if j > 0:
                neighbors.append(vertices[i * resolution + (j-1)])
            if j < resolution - 1:
                neighbors.append(vertices[i * resolution + (j+1)])
            
            if len(neighbors) >= 2:
                v1 = neighbors[0] - vertices[idx]
                v2 = neighbors[1] - vertices[idx]
                normal = np.cross(v1, v2)
                norm = np.linalg.norm(normal)
                if norm > 1e-8:
                    normals[idx] = normal / norm
    
    return normals

def normal_consistency(normals1, normals2):
    """Compute average normal consistency (dot product)."""
    dots = np.abs(np.sum(normals1 * normals2, axis=1))
    return np.mean(dots)

# Compute metrics
cd = chamfer_distance(baseline_result, neural_result)
print(f"Chamfer Distance: {cd:.6f}")

baseline_normals = compute_normals(baseline_result, 50)
neural_normals = compute_normals(neural_result, 50)
nc = normal_consistency(baseline_normals, neural_normals)
print(f"Normal Consistency: {nc:.4f}")

## 5. Measure FPS

In [ ]:
# Benchmark neural model
results = benchmark_inference(engine, n_vertices=2500, n_iterations=100)

print(f"\n=== Neural Model Performance ===")
print(f"FPS: {results['fps']:.1f}")
print(f"Latency: {results['latency_ms']:.2f} ms")
print(f"Throughput: {results['throughput_vertices_per_sec']/1e6:.2f} M vertices/sec")

# Benchmark baseline
baseline = MassSpringBaseline(test_vertices)
start = time.perf_counter()
for _ in range(100):
    _ = baseline.step()
baseline_time = time.perf_counter() - start
baseline_fps = 100 / baseline_time

print(f"\n=== Mass-Spring Baseline ===")
print(f"FPS: {baseline_fps:.1f}")
print(f"Latency: {baseline_time/100*1000:.2f} ms")

## 6. Visualization

In [ ]:
fig = plt.figure(figsize=(15, 5))

# Rest position
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(test_vertices[:, 0], test_vertices[:, 1], test_vertices[:, 2], 
            c='blue', s=1, alpha=0.5)
ax1.set_title('Rest Position')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# Baseline (mass-spring)
ax2 = fig.add_subplot(132, projection='3d')
ax2.scatter(baseline_result[:, 0], baseline_result[:, 1], baseline_result[:, 2], 
            c='green', s=1, alpha=0.5)
ax2.set_title('Mass-Spring Baseline')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')

# Neural model
ax3 = fig.add_subplot(133, projection='3d')
ax3.scatter(neural_result[:, 0], neural_result[:, 1], neural_result[:, 2], 
            c='red', s=1, alpha=0.5)
ax3.set_title('Neural Model')
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')

plt.tight_layout()
plt.savefig('cloth_comparison.png', dpi=150)
plt.show()

In [ ]:
# Summary bar chart
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Chamfer distance
axes[0].bar(['Baseline vs Neural'], [cd], color='steelblue')
axes[0].set_ylabel('Chamfer Distance')
axes[0].set_title('Geometric Accuracy')

# Normal consistency
axes[1].bar(['Baseline', 'Neural'], [1.0, nc], color=['green', 'red'])
axes[1].set_ylabel('Normal Consistency')
axes[1].set_title('Surface Quality')
axes[1].set_ylim([0, 1.1])

# FPS
axes[2].bar(['Baseline', 'Neural'], [baseline_fps, results['fps']], 
            color=['green', 'red'])
axes[2].set_ylabel('FPS')
axes[2].set_title('Inference Speed')

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150)
plt.show()

## 7. Summary

In [ ]:
print("=" * 50)
print("NIF-Cloth3D-Interactive Evaluation Summary")
print("=" * 50)
print(f"\nTest Configuration:")
print(f"  Vertices: {test_vertices.shape[0]}")
print(f"  Device: {device}")
print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")

print(f"\nMetrics:")
print(f"  Chamfer Distance: {cd:.6f}")
print(f"  Normal Consistency: {nc:.4f}")

print(f"\nPerformance:")
print(f"  Neural FPS: {results['fps']:.1f}")
print(f"  Neural Latency: {results['latency_ms']:.2f} ms")
print(f"  Baseline FPS: {baseline_fps:.1f}")
print(f"  Speedup: {results['fps']/baseline_fps:.1f}x")

# Targets
print(f"\nTargets:")
print(f"  Latency <10ms: {'✓ PASS' if results['latency_ms'] < 10 else '✗ FAIL'}")
print(f"  Normal Consistency >0.8: {'✓ PASS' if nc > 0.8 else '✗ FAIL'}")
print("=" * 50)